In [1]:
import $ivy.`org.apache.spark::spark-sql:3.5.8`
import $ivy.`org.apache.hadoop:hadoop-aws:3.3.4`
import $ivy.`com.amazonaws:aws-java-sdk-bundle:1.12.262`
import $ivy.`javax.xml.bind:jaxb-api:2.3.1`

import $ivy.$
import $ivy.$
import $ivy.$
import $ivy.$

In [2]:
import org.apache.spark.sql._
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions._

import org.apache.spark.sql._
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions._

In [3]:
val spark = SparkSession
                .builder()
                .master("local[*]")
                .appName("S3")
                .config("spark.log.level", "WARN")
                .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
                .config("spark.hadoop.fs.s3a.access.key", "admin")
                .config("spark.hadoop.fs.s3a.secret.key", "password")
                .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
                .config("spark.hadoop.fs.s3a.path.style.access", "true")
                .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
                .config("spark.hadoop.fs.s3a.fast.upload", "true")
                .getOrCreate()

import spark.implicits._

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/06 12:56:14 INFO SparkContext: Running Spark version 3.5.8
26/07/06 12:56:14 INFO SparkContext: OS info Linux, 6.12.0-211.22.1.el10_2.x86_64, amd64
26/07/06 12:56:14 INFO SparkContext: Java version 11.0.31
26/07/06 12:56:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting Spark log level to "WARN".


spark: SparkSession = org.apache.spark.sql.SparkSession@5ae501aa
import spark.implicits._

In [4]:
println(s"spark.version == ${spark.version}")

spark.version == 3.5.8


## Определяем служебные функции

In [5]:
def getRow: Row = {
    val random = new scala.util.Random
    Row(random.nextInt(5) + 1,
        random.nextInt(10000),
        random.nextFloat() * 10,
        random.nextDouble() * 10,
        if (random.nextBoolean) "Y" else "N")
}

defined function getRow

In [6]:
def getSeq(n: Int): Seq[Row] = {
    (1 to n).map { _ => getRow }
}

defined function getSeq

In [7]:
def getDF(spark: SparkSession, data: Seq[Row], schema: StructType): DataFrame = {
    spark.createDataFrame(spark.sparkContext.parallelize(data), schema)
}

defined function getDF

## Создание DataFrame

In [8]:
val schema = StructType( Array(
      StructField("vendor_id", IntegerType, nullable = true),
      StructField("trip_id", IntegerType, nullable = true),
      StructField("trip_distance", FloatType, nullable = true),
      StructField("fare_amount", DoubleType, nullable = true),
      StructField("store_and_fwd_flag", StringType, nullable = true)
    ))

schema: StructType = StructType(
  StructField("vendor_id", IntegerType, true, {}),
  StructField("trip_id", IntegerType, true, {}),
  StructField("trip_distance", FloatType, true, {}),
  StructField("fare_amount", DoubleType, true, {}),
  StructField("store_and_fwd_flag", StringType, true, {})
)

In [9]:
val data = getSeq(1000000)
val df1 = getDF(spark, data, schema)

data: Seq[Row] = Vector(
  [3,8685,6.301875,9.593874175395955,N],
  [1,7424,3.7378054,2.5625275691814506,N],
  [1,1607,1.5492857,3.7380109389146354,Y],
  [3,1826,2.417026,6.3867777318852905,Y],
  [5,3266,3.5264473,6.472560399745283,N],
  [1,6100,2.3898852,1.4547029970718295,Y],
  [1,2726,7.7786064,5.949360169488359,N],
  [2,6221,4.148371,9.735726337743994,N],
  [1,7858,6.386055,3.433037810745727,N],
  [4,3529,1.8341845,6.570693740241431,Y],
  [4,6402,8.055786,4.922485315976238,N],
  [1,9682,0.15467584,1.3721076998494985,N],
  [1,7997,8.396057,3.570269461375075,Y],
  [1,8410,6.350774,6.493814152006877,N],
  [3,5745,9.21752,6.696924800346222,Y],
  [2,1211,1.086973,1.5555351572538823,Y],
  [2,9303,5.306185,5.7159891321186045,N],
  [4,6935,0.34510314,8.815589906407526,N],
  [4,9586,2.4695814,9.094948887865144,Y],
  [5,4635,9.778384,1.1534010387383165,N],
  [4,3112,4.42202,6.683504002241642,N],
  [5,3567,4.5376215,5.919662796141017,Y],
  [5,3502,4.9241424,9.648427170418902,Y],
  [5,1664,8.9

## Запись

In [10]:
df1.write.mode(SaveMode.Overwrite).parquet("s3a://data/df1")

26/07/06 12:56:19 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/07/06 12:56:22 WARN TaskSetManager: Stage 0 contains a task of very large size (7325 KiB). The maximum recommended task size is 1000 KiB.


In [11]:
df1.write.mode(SaveMode.Append).parquet("s3a://data/df1")

26/07/06 12:56:30 WARN TaskSetManager: Stage 1 contains a task of very large size (7325 KiB). The maximum recommended task size is 1000 KiB.


## Чтение

In [12]:
val df2 = spark.read.parquet("s3a://data/df1")

df2: DataFrame = [vendor_id: int, trip_id: int ... 3 more fields]

In [13]:
df2.count()

res13: Long = 2000000L

In [14]:
df2.show()

+---------+-------+-------------+------------------+------------------+
|vendor_id|trip_id|trip_distance|       fare_amount|store_and_fwd_flag|
+---------+-------+-------------+------------------+------------------+
|        3|   8685|     6.301875| 9.593874175395955|                 N|
|        1|   7424|    3.7378054|2.5625275691814506|                 N|
|        1|   1607|    1.5492857|3.7380109389146354|                 Y|
|        3|   1826|     2.417026|6.3867777318852905|                 Y|
|        5|   3266|    3.5264473| 6.472560399745283|                 N|
|        1|   6100|    2.3898852|1.4547029970718295|                 Y|
|        1|   2726|    7.7786064| 5.949360169488359|                 N|
|        2|   6221|     4.148371| 9.735726337743994|                 N|
|        1|   7858|     6.386055| 3.433037810745727|                 N|
|        4|   3529|    1.8341845| 6.570693740241431|                 Y|
|        4|   6402|     8.055786| 4.922485315976238|            